# Problem Set 1

#NOTES
1. Assumed that 'prom_x' is an indicator for promotion or not for each product x but there are values != 0, 1 in everything except for prom_10

## 1. Sum Stats and Institutional Details

### (a) Do some brief diligence on the products and industry. What do you anticipate may be some important determinants of demand, substitution, and pricing?

### (b) Complete the table above by adding columns for the mean: market share, unit price, price/100 tablets, and unit wholesale price. Interpret any notable patterns you see in the summary statistics.

In [1]:
import numpy as np 
import pandas as pd 
import pyblp
from linearmodels.iv import IV2SLS
import statsmodels.api as sm

In [2]:
otc = pd.read_csv('OTC_Sales.csv')

#calculate total sales for each store-week pair
product_numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
otc['total_sales'] = otc[[f'sales_{number}' for number in product_numbers]].sum(axis = 1)


In [3]:
#reshape to make it easier to work with
otc = pd.wide_to_long(otc, ['sales', 'price', 'cost', 'prom'], i = ['store', 'week'], j = 'product', sep = '_').reset_index()
summary = otc.groupby('product').agg(
    price = ('price', 'mean'),
    total_sales = ('sales', 'sum'),
    wholesale_price = ('cost', 'mean')
).reset_index()

#add given information so that table is easy to read
brand_names = {
    1: 'Tylenol',
    2: 'Tylenol',
    3: 'Tylenol', 
    4: 'Advil', 
    5: 'Advil', 
    6: 'Advil', 
    7: 'Bayer', 
    8: 'Bayer', 
    9: 'Bayer', 
    10: 'Store Brand', 
    11: 'Store Brand'}

size = {
    1: 25, 
    2: 50, 
    3: 100,
    4: 25,
    5:  50,
    6: 100,
    7: 25, 
    8: 50,
    9: 100,
    10: 50, 
    11: 100
}

summary['brand'] = summary['product'].map(brand_names)
summary['size'] = summary['product'].map(size)

summary['share'] = summary['total_sales'] / (summary['total_sales'].sum())
summary['price_per_100'] = (summary['price'] / summary['size']) * 100

#reorder columns and round for clarity
summary = summary[['product', 'brand', 'size', 'share', 'price', 'price_per_100', 'wholesale_price']].round(2)
summary

,product,brand,size,share,price,price_per_100,wholesale_price
0,1,Tylenol,25,0.14,3.43,13.71,2.19
1,2,Tylenol,50,0.18,4.95,9.89,3.68
2,3,Tylenol,100,0.12,7.03,7.03,5.77
3,4,Advil,25,0.12,2.97,11.88,2.03
4,5,Advil,50,0.08,5.15,10.29,3.63
5,6,Advil,100,0.04,8.16,8.16,6.10
6,7,Bayer,25,0.04,2.67,10.70,1.85
7,8,Bayer,50,0.03,3.62,7.24,2.44
8,9,Bayer,100,0.08,3.97,3.97,3.71
9,10,Store Brand,50,0.09,1.94,3.87,0.91


## 2. Logit Demand Estimation
### Consider the utility function for product j in store-week t for consumer i: 
### $ u_{ijt} = −αp_{jt} + X_{jt}β + ξ_{jt} + ϵ_{ijt} $ (1)
### where $p_{jt}$ is price, $X_{jt}$ are other observed product characteristics, $ξ_{jt}$ are unobserved product characteristics, and $ϵ_{ijt}$ is an i.i.d. EV1 logit consumer-product-market unobservable.
### Estimate this model:

### (a) Using OLS with price, promotion, and an indicator for whether the product is a “store brand” as product characteristics.

In [4]:
#need shares for each observation
otc['share'] = otc['sales'] / otc['total_sales']

#indicator for store brand
otc['store_brand'] = np.where(otc['product'] > 9, 1, 0)
otc = otc.rename(columns = {'price': 'prices'})
otc['market_ids'] = otc['store'].astype(str) + '_' + otc['week'].astype(str)

In [5]:
#create outside shares so that shares sum to less than 1
otc['outside_share'] = 1 - otc.groupby('market_ids')['share'].transform('sum')
otc['outside_share'] = otc['outside_share'].clip(lower=1e-6)
otc['shares'] = otc['share'] - otc['outside_share']

In [6]:
# logit_formulation = pyblp.Formulation('prices + prom + store_brand')
# problem = pyblp.Problem(logit_formulation, otc)
# results = problem.solve()
# print(results)

In [7]:
exog = otc[['prices', 'prom', 'store_brand']]
otc['ln_share'] = np.log(otc['shares'])
endog = otc['ln_share']
model = sm.OLS(endog, exog)
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               ln_share   R-squared (uncentered):                   0.831
Model:                            OLS   Adj. R-squared (uncentered):              0.831
Method:                 Least Squares   F-statistic:                          6.109e+04
Date:                Thu, 30 Jan 2025   Prob (F-statistic):                        0.00
Time:                        16:50:34   Log-Likelihood:                         -58602.
No. Observations:               37290   AIC:                                  1.172e+05
Df Residuals:                   37287   BIC:                                  1.172e+05
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
prices         -0.4916      0.001   -355.310      0.000      -0.494      -0.489
prom           -0.4603      0.021    -22.330      0.000      -0.501      -0.420
store_brand    -1.2090      0.015    -81.483      0.000      -1.238      -1.180
==============================================================================
Omnibus:                      769.202   Durbin-Watson:                   1.636
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              792.450
Skew:                          -0.342   Prob(JB):                    8.35e-173
Kurtosis:                       2.796   Cond. No.                         16.3
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### (b) Using OLS with price and promotion as product characteristics and product fixed effects (where a “product” is a brand-size combination).

In [8]:
# logit_formulation = pyblp.Formulation('prices + prom', absorb='C(product)')
# problem = pyblp.Problem(logit_formulation, otc)
# results = problem.solve()
# print(results)

In [9]:
exog = otc[['prices', 'prom', 'product']]
endog = otc['ln_share']
model = sm.OLS(endog, exog)
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               ln_share   R-squared (uncentered):                   0.914
Model:                            OLS   Adj. R-squared (uncentered):              0.914
Method:                 Least Squares   F-statistic:                          1.328e+05
Date:                Thu, 30 Jan 2025   Prob (F-statistic):                        0.00
Time:                        16:50:34   Log-Likelihood:                         -45915.
No. Observations:               37290   AIC:                                  9.184e+04
Df Residuals:                   37287   BIC:                                  9.186e+04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
prices        -0.2872      0.001   -203.155      0.000      -0.290      -0.284
prom           0.2052      0.015     13.622      0.000       0.176       0.235
product       -0.2273      0.001   -222.388      0.000      -0.229      -0.225
==============================================================================
Omnibus:                      130.584   Durbin-Watson:                   2.200
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              132.421
Skew:                          -0.140   Prob(JB):                     1.76e-29
Kurtosis:                       3.083   Cond. No.                         27.6
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### (c) Estimate the models of (a) and (b) using the Hausman instrument (average price in other markets).

In [10]:
#create the Hausman instrument
def get_instruments(df):
    avg_prices = df.groupby(['store', 'week', 'product'])['prices'].mean().reset_index()

    def calculate_instrument(row):
        mask = (avg_prices['week'] == row['week']) & (avg_prices['product'] == row['product']) & (avg_prices['store'] != row['store'])
        return avg_prices.loc[mask, 'prices'].mean()

    df['instrument'] = df.apply(calculate_instrument, axis=1)

    return df

otc_iv = get_instruments(otc)

In [11]:
# #model a
# logit_formulation = pyblp.Formulation('prices + prom + store_brand')
# problem = pyblp.Problem(logit_formulation, otc)
# results = problem.solve()
# print(results)

In [12]:
dependent = otc['ln_share']
endog = otc['prices']
exog = otc[['prom', 'store_brand']]
instrument = otc['instrument']
logit_model = IV2SLS(dependent, exog, endog, instrument).fit()
logit_model.summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:               ln_share   R-squared:                      0.8309
Estimator:                    IV-2SLS   Adj. R-squared:                 0.8309
No. Observations:               37290   F-statistic:                 2.185e+05
Date:                Thu, Jan 30 2025   P-value (F-stat)                0.0000
Time:                        16:50:39   Distribution:                  chi2(3)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                              
===============================================================================
             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------
prom           -0.4550     0.0200    -22.791     0.0000     -0.4942     -0.4159
store_brand    -1.2058     0.0127    -95.279     0.0000     -1.2306     -1.1810
prices         -0.4928     0.0013    -391.01     0.0000     -0.4953     -0.4904
===============================================================================

Endogenous: prices
Instruments: instrument
Robust Covariance (Heteroskedastic)
Debiased: False
"""

In [13]:
# #model b
# logit_formulation = pyblp.Formulation('prices + prom', absorb='C(product)')
# problem = pyblp.Problem(logit_formulation, otc)
# results = problem.solve()
# print(results)

In [14]:
dependent = otc['ln_share']
endog = otc['prices']
exog = otc[['prom', 'store_brand', 'product']]
instrument = otc['instrument']
logit_model = IV2SLS(dependent, exog, endog, instrument).fit()
logit_model.summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:               ln_share   R-squared:                      0.9243
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9243
No. Observations:               37290   F-statistic:                 4.833e+05
Date:                Thu, Jan 30 2025   P-value (F-stat)                0.0000
Time:                        16:50:39   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                              
===============================================================================
             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------
prom            0.3208     0.0157     20.382     0.0000      0.2900      0.3517
store_brand     0.9926     0.0168     58.929     0.0000      0.9596      1.0257
product        -0.2957     0.0015    -193.22     0.0000     -0.2987     -0.2927
prices         -0.2405     0.0013    -179.67     0.0000     -0.2431     -0.2379
===============================================================================

Endogenous: prices
Instruments: instrument
Robust Covariance (Heteroskedastic)
Debiased: False
"""

### (d) Compute the mean own-price elasticities for all products

## 3. New Product Introduction
### The chain of stores you have data from is considering introducing a 25 tablet size bottle.

### (a) Assume WTP for the new product will be equal to average WTP of the brand name 25 tablet products, minus the “store brand” effect you estimated in the first part above. Assume there are no promotions of the new product. What will be the expected demand for the new product at a price of $2.00?

### (b) Break down the benefits and costs to the store of introducing this new format (assume wholesale price is $1.00 and there are no fixed or other variable costs of the new product introduction).

## 4. Information Intervention
### The chain of stores you have data from is considering a campaign to help educate customers that there is no efficacy difference between the brand name and “store brand” drugs.

### (a) Assume the campaign increases the WTP for the “store brands” by the full absolute value of the “store brand” effect you estimated in the first part. What is the net benefit and cost to the store? To consumers?